# Angle Sweep

Run cells top to bottom. Cell 1 launches all nodes. Cell 2 connects. Cell 3 runs the full sweep.

In [ ]:
import subprocess, os, time

SAM3_PY  = os.path.expanduser('~/sam3_env/bin/python3')
SAM3_SCR = os.path.expanduser('~/magpie_control/scripts/sam3_infer.py')
ROS      = 'source /opt/ros/humble/setup.bash && source ~/ws_ctrl/install/setup.bash'

def ros_proc(cmd, log):
    open(log, 'w').close()
    return subprocess.Popen(f'{ROS} && {cmd}', shell=True, executable='/bin/bash',
                            stdout=open(log, 'a'), stderr=subprocess.STDOUT)

print('Killing stale processes...')
for p in ['ur5_node', 'gripper_node', 'realsense2_camera', 'sam3_infer']:
    subprocess.run(f'pkill -9 -f {p}', shell=True)
subprocess.run('rm -f /tmp/sam3.sock', shell=True)
time.sleep(3)

PROCS = {}
PROCS['ur5']     = ros_proc('ros2 run magpie_control ur5_node',     '/tmp/log_ur5.txt')
PROCS['gripper'] = ros_proc('ros2 run magpie_control gripper_node', '/tmp/log_gripper.txt')
PROCS['camera']  = ros_proc(
    'ros2 run realsense2_camera realsense2_camera_node --ros-args -r __ns:=/camera/gripper_camera',
    '/tmp/log_camera.txt')
PROCS['sam3'] = subprocess.Popen(
    [SAM3_PY, SAM3_SCR, '--socket'],
    stdout=open('/tmp/log_sam3.txt', 'w'), stderr=subprocess.STDOUT)

print('Waiting for SAM3 (up to 90s)...')
for i in range(90):
    time.sleep(1)
    if os.path.exists('/tmp/sam3.sock'):
        print(f'  SAM3 ready after {i+1}s'); break
    if PROCS['sam3'].poll() is not None:
        print('  SAM3 crashed'); break
    if (i+1) % 20 == 0: print(f'  {i+1}s...')
time.sleep(3)
for n, p in PROCS.items():
    print(f'  {n:<10} {"OK" if p.poll() is None else "EXITED"}')

In [ ]:
import sys, os, ctypes, glob, time, re, json, tempfile, base64, socket as _sock
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
%matplotlib inline
plt.rcParams['figure.dpi'] = 80

for _d in ['/opt/ros/humble/lib/x86_64-linux-gnu', '/opt/ros/humble/lib',
           '/home/user/ws_ctrl/install/magpie_msgs/lib',
           '/home/user/ws_ctrl/install/magpie_control/lib']:
    for _so in sorted(glob.glob(_d + '/*.so*')):
        try: ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except Exception: pass

for _p in [
    '/opt/ros/humble/local/lib/python3.10/dist-packages',
    '/opt/ros/humble/lib/python3.10/site-packages',
    '/home/user/ws_ctrl/install/magpie_msgs/local/lib/python3.10/dist-packages',
    '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages',
    '/home/user/.local/lib/python3.10/site-packages',
    '/home/user/magpie_control/scripts',
]:
    if _p not in sys.path: sys.path.insert(0, _p)

import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy
from sensor_msgs.msg import Image as RosImage, CameraInfo
from geometry_msgs.msg import PoseStamped, Pose
from std_srvs.srv import Trigger
from cv_bridge import CvBridge
from magpie_msgs.msg import GripperState
from magpie_msgs.srv import MoveLinear, SetGripperForce, SetGripperPosition
from magpie_control import poses
from magpie_control.homog_utils import homog_xform, R_krot
from magpie_control.gripper_arc import fingertip_drop
from google import genai
from google.genai import types as gtypes
import importlib, pointcloud_utils; importlib.reload(pointcloud_utils)
from pointcloud_utils import build_segmented_pcd, denoise_pcd, analyse_pcd, grasp_rotation_matrix, smart_grasp_angle, top_layer

GEMINI_KEY     = os.environ.get('GEMINI_API_KEY', '')
SAM3_SOCK      = '/tmp/sam3.sock'
GRIPPER_LEN    = 0.231
HARD_FLOOR_Z   = 0.025
APPROACH_H     = 0.10
TABLE_Z        = None   # set after running measure cell, or leave None for auto-measure
PLACE_HEIGHT_OFFSET = 0.005   # m above table when placing
_TCP_TO_CAM    = homog_xform(R_krot([0, 0, 1], np.pi/2), [0, 0, 0.120])
gc             = genai.Client(api_key=GEMINI_KEY)

img_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.TRANSIENT_LOCAL)
inf_qos = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                     durability=DurabilityPolicy.VOLATILE)

def _quat_to_rv(w,x,y,z):
    a=2.*np.arccos(np.clip(w,-1,1)); s=np.sin(a/2)
    return np.zeros(3) if s<1e-10 else a*np.array([x,y,z])/s
def _rv_to_quat(rv):
    a=np.linalg.norm(rv)
    if a<1e-10: return(1.,0.,0.,0.)
    ax=rv/a; return(np.cos(a/2),ax[0]*np.sin(a/2),ax[1]*np.sin(a/2),ax[2]*np.sin(a/2))
def _mat_to_pose(mat):
    v=poses.pose_mtrx_to_vec(np.array(mat)); w,x,y,z=_rv_to_quat(np.array(v[3:]))
    p=Pose(); p.position.x,p.position.y,p.position.z=v[0],v[1],v[2]
    p.orientation.w,p.orientation.x,p.orientation.y,p.orientation.z=w,x,y,z; return p

class Demo(Node):
    def __init__(self):
        super().__init__('angle_sweep')
        self.bridge=CvBridge(); self.color=self.depth=self.caminfo=self.tcp=self.gs=None
        NS='/camera/gripper_camera/camera'
        self.create_subscription(RosImage,NS+'/color/image_raw',
            lambda m: setattr(self,'color',self.bridge.imgmsg_to_cv2(m,'rgb8')),img_qos)
        self.create_subscription(RosImage,NS+'/depth/image_rect_raw',
            lambda m: setattr(self,'depth',self.bridge.imgmsg_to_cv2(m,'passthrough')),img_qos)
        self.create_subscription(CameraInfo,NS+'/color/camera_info',
            lambda m: setattr(self,'caminfo',m),inf_qos)
        self.create_subscription(GripperState,'/gripper/state',
            lambda m: setattr(self,'gs',m),1)
        self.create_subscription(PoseStamped,'/arm/tcp_pose',self._tcp_cb,1)
        self.mv =self.create_client(MoveLinear,'/arm/move_l')
        self.tch=self.create_client(Trigger,'/arm/teach_mode')
        self.opn=self.create_client(Trigger,'/gripper/open')
        self.cls=self.create_client(Trigger,'/gripper/close')
        self.frc=self.create_client(SetGripperForce,'/gripper/set_force')
        self.pos=self.create_client(SetGripperPosition,'/gripper/set_position')
    def _tcp_cb(self,m):
        p=m.pose; rv=_quat_to_rv(p.orientation.w,p.orientation.x,p.orientation.y,p.orientation.z)
        self.tcp=poses.pose_vec_to_mtrx([p.position.x,p.position.y,p.position.z,*rv])
    def spin(self,n=8):
        for _ in range(n): rclpy.spin_once(self,timeout_sec=0.15)
    def _call(self,client,req,timeout=30.):
        client.wait_for_service(timeout_sec=4.)
        fut=client.call_async(req)
        rclpy.spin_until_future_complete(self,fut,timeout_sec=timeout)
        return fut.result()
    def move(self,mat,spd=0.08):
        r=MoveLinear.Request(); r.target_pose=_mat_to_pose(mat)
        r.speed=spd; r.acceleration=0.2; r.async_mode=False
        resp=self._call(self.mv,r)
        if not resp.success: raise RuntimeError(resp.message)
    def open_g(self):  return self._call(self.opn,Trigger.Request())
    def close_g(self): return self._call(self.cls,Trigger.Request())
    def set_force(self,n):
        r=SetGripperForce.Request(); r.max_force=float(n); return self._call(self.frc,r)
    def set_pos(self,mm):
        r=SetGripperPosition.Request(); r.position=float(max(0,mm)); return self._call(self.pos,r)
    def unteach(self):
        if not self.tch.wait_for_service(timeout_sec=2.): return
        r=self._call(self.tch,Trigger.Request())
        if r and 'enabled' in r.message.lower(): self._call(self.tch,Trigger.Request())
    def wait_sensors(self,timeout=20.):
        t0=time.time()
        while time.time()-t0<timeout:
            rclpy.spin_once(self,timeout_sec=0.15)
            if all(v is not None for v in [self.color,self.depth,self.caminfo,self.tcp]): return True
        return False

def sam3_query(img_rgb, query, sock_path=SAM3_SOCK):
    tmp=tempfile.mktemp(suffix='.jpg')
    cv2.imwrite(tmp,cv2.cvtColor(img_rgb,cv2.COLOR_RGB2BGR))
    try:
        with _sock.socket(_sock.AF_UNIX,_sock.SOCK_STREAM) as s:
            s.connect(sock_path)
            s.sendall((json.dumps({'image':tmp,'query':query})+'\n').encode())
            raw=b''
            while True:
                chunk=s.recv(65536)
                if not chunk: break
                raw+=chunk
        d=json.loads(raw.decode().strip())
        if 'error' in d: raise RuntimeError(d['error'])
        boxes=np.array(d['boxes'],dtype=float); scores=np.array(d['scores'],dtype=float)
        mask=None
        if d.get('mask_b64') and len(boxes)>0:
            raw2=base64.b64decode(d['mask_b64']); h,w=d['mask_shape']
            mask=np.frombuffer(raw2,dtype=np.uint8).reshape(h,w).astype(bool)
        return boxes,scores,mask
    finally:
        if os.path.exists(tmp): os.unlink(tmp)

try: rclpy.init()
except RuntimeError: pass
try: node.destroy_node()
except Exception: pass
node=Demo()
ok=node.wait_sensors(20.)
home=node.tcp.copy()
print('Sensors ready:', ok, '| home z=', f'{home[2,3]*1000:.1f}mm')

In [ ]:
import sys, os
for _p in ['/home/user/magpie_control/scripts',
           '/opt/ros/humble/local/lib/python3.10/dist-packages',
           '/home/user/ws_ctrl/install/magpie_control/lib/python3.10/site-packages']:
    if _p not in sys.path: sys.path.insert(0, _p)

import importlib, pointcloud_utils; importlib.reload(pointcloud_utils)
from pointcloud_utils import build_segmented_pcd, denoise_pcd, analyse_pcd, grasp_rotation_matrix, smart_grasp_angle, top_layer, grasp_rotation_matrix as _grm

GP = globals().get('GP')  # dict from DeliGrasp cell, or None if not run
SWEEP_ANGLES = list(range(0, 91, 10))   # [0, 10, 20, ..., 90]
sweep_log    = []

# View pose — straight-down orientation at home XY, fixed scan height
# Arm moves here before every snapshot so camera always looks straight down
VIEW_Z_OFFSET = 0.05   # m above home height — adjust if needed
view_pose        = home.copy()
view_pose[:3,:3] = _grm(0.)   # straight-down, wrist at 0°
view_pose[2,3]   = home[2,3] + VIEW_Z_OFFSET

prev_place_angle = None   # tracks what angle we placed at last iteration

for sweep_i, place_angle in enumerate(SWEEP_ANGLES):
    print(f'\n=== Sweep {sweep_i+1}/{len(SWEEP_ANGLES)}  place_angle={place_angle}° ===')

    # ── Open gripper fully before any arm movement ────────────────────────────
    node.open_g(); time.sleep(1.0)
    # ── Move to view position ─────────────────────────────────────────────────
    node.unteach()
    node.move(view_pose, spd=0.08); time.sleep(0.5); node.spin(6)
    print(f'  At view pose — taking snapshot')

    node.spin(8)
    snap_color=node.color.copy(); snap_depth=node.depth.copy()
    snap_tcp=node.tcp.copy(); snap_ci=node.caminfo; home_sw=snap_tcp.copy()

    _, _buf=cv2.imencode('.jpg',cv2.cvtColor(snap_color,cv2.COLOR_RGB2BGR))
    ACTIVE=' '.join(gc.models.generate_content(
        model='gemini-2.5-flash',
        contents=[gtypes.Part.from_bytes(data=_buf.tobytes(),mime_type='image/jpeg'),
                  'What is the main graspable object? Reply 2-4 words only, no punctuation.']
    ).text.strip().lower().split()[:4])
    print(f'  Object: "{ACTIVE}"')

    det_boxes,det_scores,det_mask=sam3_query(snap_color,ACTIVE)
    if len(det_scores)==0 or det_mask is None:
        print('  No detection — skipping'); sweep_log.append({'sweep':sweep_i+1,'place_angle':place_angle,'st':'no_detection'}); continue
    best_i=int(np.argmax(det_scores))
    print(f'  SAM3 score={det_scores[best_i]:.3f}')

    pts_raw,pcd_raw=build_segmented_pcd(det_mask,snap_depth,snap_ci.k,snap_tcp,_TCP_TO_CAM)
    _,pts_cln=denoise_pcd(pcd_raw)
    if len(pts_cln)<3: print('  Too few points — skipping'); sweep_log.append({'sweep':sweep_i+1,'place_angle':place_angle,'st':'no_pcd'}); continue
    pts_top=top_layer(pts_cln)           # top Z layer for angle only
    pca=analyse_pcd(pts_top); obj_w_mm=pca['extent_m'][1]*1000.
    ang,strategy,reason=smart_grasp_angle(pca, object_name=ACTIVE, image_rgb=snap_color, gemini_client=gc, mask=det_mask)
    print(f'  Detected angle={ang:.1f}°  expected={prev_place_angle if prev_place_angle is not None else "N/A (first run)"}°  [{strategy}]')

    ys,xs=np.where(det_mask); u,v=int(xs.mean()),int(ys.mean())
    valid=snap_depth[det_mask].astype(float); valid=valid[valid>0]
    dm=float(np.median(valid))/1000.
    fx,fy=snap_ci.k[0],snap_ci.k[4]; cx,cy=snap_ci.k[2],snap_ci.k[5]
    T=snap_tcp@_TCP_TO_CAM
    p_obj=(T@np.array([(u-cx)*dm/fx,(v-cy)*dm/fy,dm,1.]))[:3]
    _tz=TABLE_Z if TABLE_Z is not None else HARD_FLOOR_Z
    grasp_off=float(np.clip(max(p_obj[2]-_tz,0.010)/2.,0.010,0.040))

    afz=p_obj[2]+APPROACH_H; gfz=p_obj[2]-grasp_off
    if afz<HARD_FLOOR_Z or gfz<HARD_FLOOR_Z:
        print('  Safety FAIL — skipping'); sweep_log.append({'sweep':sweep_i+1,'place_angle':place_angle,'st':'safety_fail'}); continue


    ap=home_sw.copy(); ap[0,3]=p_obj[0]; ap[1,3]=p_obj[1]; ap[2,3]=p_obj[2]+APPROACH_H+GRIPPER_LEN
    ap_rot=ap.copy(); ap_rot[:3,:3]=grasp_rotation_matrix(ang)
    grasp_pose=ap_rot.copy(); grasp_pose[2,3]=gfz+GRIPPER_LEN

    node.open_g(); time.sleep(0.5)

    node.move(ap, spd=0.08); time.sleep(0.4); node.spin(4)
    node.move(ap_rot, spd=0.05); time.sleep(0.4); node.spin(4)
    node.move(grasp_pose, spd=0.05); time.sleep(0.5); node.spin(4)

    if 'GP' in dir() and GP is not None and GP.get('mass_g') and GP.get('mu'):
        F_min       = (GP['mass_g'] / 1000. * 9.81) / (2. * GP['mu'])
        force_cap   = GP.get('force_cap', 16.)
        cf          = float(np.clip(max(GP['initial_force'], F_min * 1.2, 5.), 5., force_cap))
        slip_thresh = float(max(0.15, min(F_min * 0.9, force_cap * 0.8)))
        add         = GP['add_force']
    else:
        F_min = 0.; cf = 5.; slip_thresh = 0.15; add = 1.; force_cap = 16.

    pre_ap = float(np.clip(obj_w_mm + 5., 10., 100.))
    node.set_pos(pre_ap); time.sleep(0.8)

    grasp_log = []
    for attempt in range(6):
        target_ap = GP['aperture_mm'] if 'GP' in dir() and GP is not None and GP.get('aperture_mm') else 0.
        node.set_force(cf)
        if target_ap > 1.:
            node.set_pos(target_ap); time.sleep(2.0)
            for _ in range(6): rclpy.spin_once(node, timeout_sec=0.15)
            s = node.gs
            if s is not None and s.force < slip_thresh:
                node.close_g(); time.sleep(2.0)
        else:
            node.close_g(); time.sleep(3.0)
        for _ in range(8): rclpy.spin_once(node, timeout_sec=0.15)
        s = node.gs
        if s is None: break
        status = 'contact' if s.force >= slip_thresh else 'slip'
        print(f'  attempt {attempt+1}: ap={s.position:.1f}mm F={s.force:.3f}N -> {status}')
        grasp_log.append({'a': attempt+1, 'ap': s.position, 'f': s.force, 'st': status})
        if status == 'contact': break
        deficit = max(0., slip_thresh - s.force)
        cf = min(force_cap, cf + max(add, deficit + 0.5))
        if target_ap > 1.: target_ap = max(1., target_ap - GP.get('closure_mm', 5.))

    node.move(ap_rot, spd=0.05); time.sleep(0.5); node.spin(8)
    s_lift      = node.gs
    force_ok    = s_lift is not None and s_lift.force >= slip_thresh
    aperture_ok = s_lift is not None and s_lift.position >= obj_w_mm * 0.4
    held        = force_ok and aperture_ok
    print(f'  Pickup: {"HELD" if held else "DROPPED"}  F={s_lift.force:.3f}N  ap={s_lift.position:.1f}mm')

    if held:
        place_rot = ap_rot.copy(); place_rot[:3, :3] = grasp_rotation_matrix(float(place_angle))
        node.move(place_rot, spd=0.05); time.sleep(0.4); node.spin(4)
        place_z = place_rot.copy(); place_z[2, 3] = grasp_pose[2, 3]
        node.move(place_z, spd=0.04); time.sleep(0.4); node.spin(4)
        node.open_g(); time.sleep(0.5)
        node.move(place_rot, spd=0.05); time.sleep(0.4); node.spin(4)
        print(f'  Placed at {place_angle}°')
    else:
        node.open_g()

    node.move(home, spd=0.10); time.sleep(0.5); node.spin(4)
    last=grasp_log[-1] if grasp_log else {}
    sweep_log.append({'sweep':sweep_i+1,'place_angle':place_angle,
                      'expected_angle':prev_place_angle,'detected_angle':ang,
                      'strategy':strategy,'held':held,'attempts':len(grasp_log),
                      'force':last.get('f',0.),'aperture':last.get('ap',0.)})
    prev_place_angle = place_angle

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n=== SWEEP SUMMARY ===')
print(f'{"#":<4} {"Placed@":<9} {"Expected°":<11} {"Detected°":<11} {"Strategy":<12} {"Held":<7} {"Force"}')
for r in sweep_log:
    if 'detected_angle' not in r:
        print(f'{r["sweep"]:<4} {r["place_angle"]:<9} {"-":<11} {"-":<11} {"-":<12} {r["st"]}')
        continue
    exp = f'{r["expected_angle"]:.1f}' if r["expected_angle"] is not None else "N/A"
    print(f'{r["sweep"]:<4} {r["place_angle"]:<9} {exp:<11} {r["detected_angle"]:.1f}°{"":<6} {r["strategy"]:<12} {str(r["held"]):<7} {r["force"]:.3f}N')

# ── Plot ──────────────────────────────────────────────────────────────────────
valid=[r for r in sweep_log if 'detected_angle' in r and r['expected_angle'] is not None]
if valid:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))

    # Expected vs detected angle
    exp_angs  = [r['expected_angle'] for r in valid]
    det_angs  = [r['detected_angle']  for r in valid]
    axes[0].plot([0,90],[0,90],'k--',lw=1,alpha=0.4,label='ideal')
    axes[0].scatter(exp_angs, det_angs, c=['limegreen' if r['held'] else 'tomato' for r in valid], s=80, zorder=3)
    axes[0].set_xlabel('Expected angle (°)'); axes[0].set_ylabel('Detected angle (°)')
    axes[0].set_title('Expected vs Detected Angle'); axes[0].legend()

    # Force per angle coloured by held/dropped
    angles=[r['place_angle'] for r in [x for x in sweep_log if 'held' in x]]
    forces=[r['force']       for r in [x for x in sweep_log if 'held' in x]]
    cols  =['limegreen' if r['held'] else 'tomato' for r in [x for x in sweep_log if 'held' in x]]
    axes[1].bar(angles,forces,width=7,color=cols)
    axes[1].set_xlabel('Place angle (°)'); axes[1].set_ylabel('Force (N)'); axes[1].set_title('Grasp Force per Angle')
    axes[1].set_xticks(SWEEP_ANGLES)

    # Strategy breakdown
    strategies = {}
    for r in sweep_log:
        if 'strategy' in r: strategies[r['strategy']] = strategies.get(r['strategy'],0)+1
    axes[2].bar(strategies.keys(), strategies.values(), color='steelblue')
    axes[2].set_title('Strategy Distribution'); axes[2].set_ylabel('Count')

    plt.suptitle(f'Angle Sweep — "{ACTIVE}"'); plt.tight_layout(); plt.show()


In [ ]:
# ── Auto-append sweep results to tests/test_log.md ───────────────────────────
from datetime import date
import os

log_path = os.path.expanduser('~/magpie_control/tests/test_log.md')
today    = date.today().isoformat()

valid  = [r for r in sweep_log if 'held' in r]
n_pass = sum(1 for r in valid if r['held'])
n_total = len(valid)
skipped = len(sweep_log) - n_total

def fmt_row(r):
    exp = '-' if r['expected_angle'] is None else f'{r["expected_angle"]}deg'
    return (f'| {r["sweep"]} | {r["place_angle"]}deg | {exp} | '
            f'{r["detected_angle"]:.1f}deg | {r["strategy"]} | '
            f'{"HELD" if r["held"] else "DROPPED"} | {r["force"]:.3f}N |')

rows = '\n'.join(fmt_row(r) for r in valid)
angles_str = ', '.join(str(a) for a in SWEEP_ANGLES)

entry = (
    '\n---\n\n'
    f'## Angle Sweep — {ACTIVE}\n'
    f'**Date:** {today}\n'
    f'**Notebook:** `angle_sweep.ipynb`\n'
    f'**Object:** {ACTIVE}\n'
    f'**Result:** {n_pass}/{n_total} pickups HELD  ({skipped} skipped)\n'
    f'**Sweep angles:** {angles_str}deg\n\n'
    '| # | Place | Expected | Detected | Strategy | Result | Force |\n'
    '|---|-------|----------|----------|----------|--------|-------|\n'
    + rows + '\n\n'
    '**Notes:** \n'
)

with open(log_path, 'a') as f:
    f.write(entry)

print(f'Appended to {log_path}')
print(f'Result: {n_pass}/{n_total} HELD  ({skipped} skipped)')